# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

## Exam Instructions:
1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## Student Information

In [ ]:
# Fill in your information
name = "Sadat Maliha Mashiat"           # Write your full name here
email = "sadatmaliha703@gmail.com"          # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Sadat Maliha Mashiat
Email : sadatmaliha703@gmail.com


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.linear_model import LinearRegression, SGDRegressor

---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [ ]:
# Question 1

df = pd.read_csv('house_price_regression_dataset.csv')

display(df.shape)

display(df.head(10))

display(df.sample(5))



(1000, 8)

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
5,3944,5,3,1990,2.475930,2,8,8.797970e+05
6,3671,1,2,2012,4.911960,0,1,8.144279e+05
7,3419,1,1,1972,2.805281,1,1,7.034131e+05
8,630,3,3,1997,1.014286,1,8,1.738750e+05
9,2185,4,2,1981,3.941604,2,5,5.041765e+05


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
963,3554,2,2,1994,0.800122,1,9,734201.461578
155,2233,5,3,2010,4.106066,2,5,573797.939307
307,4236,4,3,1970,1.284095,1,2,872676.434152
581,4646,1,2,2016,1.128195,0,9,965317.506102
837,3391,1,3,1977,1.455503,1,3,694917.300875


---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [ ]:
#Question 2

#Train_test split
X = df.drop('House_Price', axis = 1)
Y = df['House_Price']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

display(X_train.shape)
display(X_test.shape)

display(Y_train.shape)
display(Y_test.shape)


#Drop columns with more than 50% missing values
df = df.dropna(axis = 1, thresh = len(df) * 0.5)


#Imputation (But no missing value, no categorical feature)
imputer_transformer = ColumnTransformer(
    transformers = [
        ('all_mean', SimpleImputer(missing_values = np.nan , strategy = 'mean'), ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size', 'Garage_Size', 'Neighborhood_Quality'])
    ],

    remainder = 'passthrough',
    verbose_feature_names_out = False
)

imputer_transformer.set_output(transform = 'pandas')

imputer_transformer.fit(X_train)

X_train = imputer_transformer.transform(X_train)
X_test = imputer_transformer.transform(X_test)





(800, 7)

(200, 7)

(800,)

(200,)

---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [ ]:
#Question 3

X = df['Square_Footage']
Y = df['House_Price']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

X_train = X_train.values
X_test = X_test.values

Y_train = Y_train.values
Y_test = Y_test.values

# slop, m = w (weight)
# intercept, c = b (bias)

def make_prediction(X, y, w, b):
    m = X.shape[0]

    pred_list = np.zeros((m,))

    for i in range(m):
        pred_list[i] = w * X[i] + b

    return pred_list

def compute_cost(X, y, w, b):
    m = X.shape[0]

    cost = 0.0

    for i in range(m):

        prediction = w * X[i] + b

        error = prediction - y[i]

        error_squared = error ** 2

        cost = cost + error_squared

    cost = cost / (2 * m)

    return cost

def calculate_gradient(X, y, w, b):
    m = X.shape[0]

    dj_dw = 0.0
    dj_db = 0.0

    for i in range(m):

        prediction = w * X[i] + b

        error = prediction - y[i]

        dj_dw = dj_dw + error * X[i]

        dj_db = dj_db + error

    dj_dw = dj_dw / m
    dj_db = dj_db / m

    return dj_dw, dj_db

def gradient_descent(X, y, w_input, b_input, max_iter, alpha = 0.01):

    w = w_input
    b = b_input

    cost_memo = []
    iteration = []

    for i in range(max_iter):
        dj_dw, dj_db = calculate_gradient(X, y, w, b)

        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        cost = compute_cost(X, y, w, b)

        cost_memo.append(cost)
        iteration.append(i)

        if i % 100 == 0:
           print(f"Iteration {i}:Cost {cost:0.4f}, w:{w:0.4f}, b:{b:0.4f}")

    return w, b, cost_memo, iteration

w_initial = 0.0
b_initial = 0.0

w, b, cost_memo, iteration = gradient_descent(X_train, Y_train, w_initial, b_initial, max_iter = 10000, alpha = 0.01)

Y_pred = make_prediction(X_test, Y_test, w, b)

print("m (Slope) =", w)

print("c (Intercept) =", b)

for i in range(10):

    print(f"Square_Footage:{X_test[i]:.0f} Actual:{Y_test[i]:.2f} Predicted:{Y_pred[i]:.2f}")








Iteration 0:Cost 2002791506812178268160.0000, w:20549781.5061, b:6185.7605
Iteration 100:Cost nan, w:nan, b:nan


/tmp/ipykernel_677/4122249696.py:38: RuntimeWarning: overflow encountered in scalar power
  error_squared = error ** 2
/tmp/ipykernel_677/4122249696.py:58: RuntimeWarning: overflow encountered in scalar multiply
  dj_dw = dj_dw + error * X[i]
/tmp/ipykernel_677/4122249696.py:78: RuntimeWarning: invalid value encountered in scalar subtract
  w = w - alpha * dj_dw


Iteration 200:Cost nan, w:nan, b:nan
Iteration 300:Cost nan, w:nan, b:nan
Iteration 400:Cost nan, w:nan, b:nan
Iteration 500:Cost nan, w:nan, b:nan
Iteration 600:Cost nan, w:nan, b:nan
Iteration 700:Cost nan, w:nan, b:nan
Iteration 800:Cost nan, w:nan, b:nan
Iteration 900:Cost nan, w:nan, b:nan
Iteration 1000:Cost nan, w:nan, b:nan
Iteration 1100:Cost nan, w:nan, b:nan
Iteration 1200:Cost nan, w:nan, b:nan
Iteration 1300:Cost nan, w:nan, b:nan
Iteration 1400:Cost nan, w:nan, b:nan
Iteration 1500:Cost nan, w:nan, b:nan
Iteration 1600:Cost nan, w:nan, b:nan
Iteration 1700:Cost nan, w:nan, b:nan
Iteration 1800:Cost nan, w:nan, b:nan
Iteration 1900:Cost nan, w:nan, b:nan
Iteration 2000:Cost nan, w:nan, b:nan
Iteration 2100:Cost nan, w:nan, b:nan
Iteration 2200:Cost nan, w:nan, b:nan
Iteration 2300:Cost nan, w:nan, b:nan
Iteration 2400:Cost nan, w:nan, b:nan
Iteration 2500:Cost nan, w:nan, b:nan
Iteration 2600:Cost nan, w:nan, b:nan
Iteration 2700:Cost nan, w:nan, b:nan
Iteration 2800:Cost 

---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [ ]:
#Question 4

encoder_scaler = ColumnTransformer(
    transformers = [
        ('all_encoder', OneHotEncoder(sparse_output = False, drop = 'first'), ['Neighborhood_Quality']),
        ('all_scaler', StandardScaler(), ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms'])
    ],

    remainder = 'passthrough',
    verbose_feature_names_out = False
)

encoder_scaler.set_output(transform = 'pandas')

encoder_scaler.fit(X_train)

X_train = encoder_scaler.transform(X_train)
X_test = encoder_scaler.transform(X_test)

display(X_train.shape)
display(X_test.shape)



(800, 15)

(200, 15)

## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [ ]:
# Question 5

#Preprocessing numerical columns (no categorical columns)
numerical_col = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size', 'Garage_Size', 'Neighborhood_Quality']

numerical_transformers = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'mean')),
        ('scaler', StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('numerical', numerical_transformers, numerical_col)
    ],

    remainder = "passthrough"
)

#Train_test split
X = df.drop('House_Price', axis = 1)
Y = df['House_Price']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

#SGD Regressor
SGD_pipe = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model' , SGDRegressor())
    ]
)

SGD_pipe.fit(X_train, Y_train)

Y_pred = SGD_pipe.predict(X_test)

print(f"RMSE:{round(root_mean_squared_error(Y_test, Y_pred), 4)}")
print(f"R2:{round(r2_score(Y_test, Y_pred), 4)}")

actual_vs_predicted = pd.DataFrame({
    'Actual': Y_test.iloc[:10].values,
    'Predicted': Y_pred[:10]
})

print(actual_vs_predicted)




RMSE:10078.8008
R2:0.9984
         Actual     Predicted
0  9.010005e+05  8.686662e+05
1  4.945375e+05  4.903068e+05
2  9.494042e+05  9.455305e+05
3  1.040389e+06  1.033377e+06
4  7.940100e+05  7.767378e+05
5  7.240336e+05  7.324217e+05
6  9.984392e+05  9.949785e+05
7  9.097134e+05  8.850929e+05
8  7.926815e+05  7.967471e+05
9  9.474908e+05  9.317846e+05


---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [ ]:
# Question 6

#Preprocessing numerical columns (no categorical columns)
numerical_col = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size', 'Garage_Size', 'Neighborhood_Quality']

numerical_transformers = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'mean')),
        ('scaler', StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('numerical', numerical_transformers, numerical_col)
    ],

    remainder = "passthrough"
)

#Train_test split
X = df.drop('House_Price', axis = 1)
Y = df['House_Price']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

#Linear Regression
LR_pipe = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model' , LinearRegression())
    ]
)

LR_pipe.fit(X_train, Y_train)

Y_pred = LR_pipe.predict(X_test)

print(f"RMSE:{round(root_mean_squared_error(Y_test, Y_pred), 4)}")
print(f"R2:{round(r2_score(Y_test, Y_pred), 4)}")


actual_vs_predicted = pd.DataFrame({
    'Actual': Y_test.iloc[:10].values,
    'Predicted': Y_pred[:10]
})

print(actual_vs_predicted)








RMSE:10071.4844
R2:0.9984
         Actual     Predicted
0  9.010005e+05  8.686871e+05
1  4.945375e+05  4.903379e+05
2  9.494042e+05  9.456671e+05
3  1.040389e+06  1.033403e+06
4  7.940100e+05  7.766988e+05
5  7.240336e+05  7.324441e+05
6  9.984392e+05  9.950520e+05
7  9.097134e+05  8.851748e+05
8  7.926815e+05  7.967305e+05
9  9.474908e+05  9.317411e+05


---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [ ]:
# Question 7

df = df.sample(n = 100, random_state = 42)

X = df[['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']].values
Y = df['House_Price'].values

X = np.insert(X, 0, 1, axis = 1)

X_T = X.T
result1 = np.dot(X_T, X)
result1_inv = np.linalg.inv(result1)
result2 = np.dot(X_T, Y)
theta = np.dot(result1_inv, result2)

print(theta)






[20888.25945218   202.30820792  8303.2810412   3020.32852173]
